<img src="https://raw.githubusercontent.com/carubbi/MQ/b58af070c45fb17b77640c06c0016859ad366b4d/notebooks/assets/imgs/UNIFOR_logo.png" width="400">
<br>
<b>
<font size="6" face="arial" color="blue">
    Graduação em Ciência da Computação
</font>
</b>
<br>
<b>
<font size="4" face="arial">
    Disciplina: Métodos Quantitativos em Computação
</font>
</b>

**Orientador: Prof. Me. Ricardo Carubbi** <br>
*Docente da Graduação e Pós-Graduação em Ciência de Dados e Inteligência Artificial*<br>
*Laboratório de Ciência de Dados e Inteligência Artificial*<br>
*Universidade de Fortaleza*<br>
**Discentes:**<br>
Arthur Ramos Souza - 2513831<br>
Samylla Miguel Barbosa - 2517429

#AP 1 — AMES HOUSING
Como se distribuem e variam os preços de venda de imóveis residenciais?

# 1. Importação e rastreabilidade


## 1.1 Bibliotecas


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## 1.2 Fonte dos dados


Arquivo `AmesHousing.csv`, disponibilizado no [Kaggle](https://www.kaggle.com/datasets/shashanknecrothapa/ames-housing-dataset), com dados de Dean De Cock. A leitura usa diretamente a URL indicada, separador vírgula e codificação UTF-8.


In [ ]:
url = "https://www.kaggle.com/api/v1/datasets/download/shashanknecrothapa/ames-housing-dataset/AmesHousing.csv"
dataframe = pd.read_csv(url, sep=",", encoding="utf-8")


## 1.3 Dimensões


In [ ]:
print("Linhas:", dataframe.shape[0])
print("Colunas:", dataframe.shape[1])


## 1.4 Unidade de análise


Cada linha representa um registro de venda de imóvel residencial em Ames, Iowa, Estados Unidos, entre 2006 e 2010. São 2.930 registros e 82 colunas na base de referência; os preços estão em dólares nominais da época da venda.


## 1.5 Inspeção inicial


In [ ]:
dataframe.head()


In [ ]:
print(dataframe.columns.tolist())


## 1.6 Variáveis selecionadas


Os nomes originais são preservados. A variável B é qualidade da cozinha — Kitchen Quality (`Kitchen Qual`); `KitchenQual` é a grafia usada na documentação, e `Kitchen Qual` é a coluna deste CSV.


In [ ]:
cols_sel = ["SalePrice", "Gr Liv Area", "Yr Sold", "Kitchen Qual", "Fireplaces",
            "Mas Vnr Area", "Year Remod/Add", "Year Built"]
df_original = dataframe.copy()
df = dataframe[cols_sel].copy()

nomes = ["Preço de venda", "Área habitável acima do solo", "Ano da venda",
         "Qualidade da cozinha (Kitchen Quality)", "Quantidade de lareiras",
         "Área de revestimento de alvenaria", "Ano de reforma/ampliação", "Ano de construção"]
unidades = ["US$", "ft²", "ano", "categoria ordinal", "lareiras", "ft²", "ano", "ano"]

variaveis = pd.DataFrame({"Nome legível": nomes, "Nome original": cols_sel,
                         "Nome utilizado": cols_sel, "Unidade": unidades})
variaveis


# 2. Qualidade e preparação


## 2.1 Tipos e ausências


In [ ]:
qualidade = pd.DataFrame({"Tipo original": df.dtypes,
                          "Válidos": df.count(),
                          "Ausentes": df.isna().sum(),
                          "Ausentes (%)": df.isna().sum() / len(df) * 100})
qualidade.round(2)


As colunas numéricas já foram lidas como números. `Kitchen Qual` contém códigos de categorias; sua versão agrupada será ordinal. Somente `Mas Vnr Area` tem ausências nas oito colunas selecionadas: 23 registros.


## 2.2 Duplicidades e consistência


In [ ]:
verificacoes = pd.DataFrame({
    "Verificação": ["Linhas completas duplicadas", "Identificadores Order repetidos",
                    "Preço não positivo", "Área não positiva", "Lareiras negativas",
                    "Revestimento com área negativa", "Reforma anterior à construção",
                    "Construção posterior à venda", "Reforma posterior à venda"],
    "Quantidade": [df_original.duplicated().sum(), df_original["Order"].duplicated().sum(),
                   (df["SalePrice"] <= 0).sum(), (df["Gr Liv Area"] <= 0).sum(),
                   (df["Fireplaces"] < 0).sum(), (df["Mas Vnr Area"] < 0).sum(),
                   (df["Year Remod/Add"] < df["Year Built"]).sum(),
                   (df["Year Built"] > df["Yr Sold"]).sum(),
                   (df["Year Remod/Add"] > df["Yr Sold"]).sum()]
})
verificacoes


Não há linhas completas duplicadas nem identificadores `Order` repetidos. Há quatro registros com alguma inconsistência entre anos; as três verificações de datas podem contar o mesmo registro.


In [ ]:
datas_inconsistentes = ((df["Year Remod/Add"] < df["Year Built"]) |
                       (df["Year Built"] > df["Yr Sold"]) |
                       (df["Year Remod/Add"] > df["Yr Sold"]))
df_original.loc[datas_inconsistentes, ["Order", "Year Built", "Year Remod/Add", "Yr Sold"]]


## 2.3 Significado e tratamentos


Segundo a [documentação de Ames Housing](https://jse.amstat.org/v19n3/decock/DataDocumentation.txt), `Fireplaces` conta lareiras e zero significa nenhuma; `Mas Vnr Area` mede revestimento em ft² e ausência de informação não significa área zero. `Year Remod/Add` repete `Year Built` quando não há reforma/ampliação registrada.


Os 23 valores ausentes de `Mas Vnr Area` permanecem ausentes. As datas inconsistentes são preservadas por falta de evidência para corrigi-las e precisam de revisão antes de cálculos de idade/reforma na AP2. Nenhuma linha é removida, e `df_original` conserva a cópia lógica completa.


In [ ]:
preco = df["SalePrice"]
area = df["Gr Liv Area"]
cozinha = df["Kitchen Qual"]


# 3. Análise univariada


## 3.1 Preparação das medidas


As medidas individuais têm cálculos manuais e com pandas em células separadas. Todas as tabelas usam apenas pandas; `dropna()` desconsidera ausências somente no cálculo, e a variância e o desvio-padrão usam o denominador n − 1.


In [ ]:
# Define a preparação comum que remove ausências e verifica o tamanho mínimo.
def _validos(valores: pd.Series, minimo: int = 1) -> pd.Series:
    """Remove ausências de uma série sem convertê-la."""
    if not isinstance(valores, pd.Series):
        raise TypeError(
            "valores deve ser pandas.Series numérica ou booleana; "
            "converta listas antes com pd.Series(valores)."
        )
    serie = valores.dropna()
    if len(serie) < minimo:
        raise ValueError(
            f"São necessárias pelo menos {minimo} observações válidas."
        )
    return serie


## 3.2 Quantidade, mínimo e máximo


A quantidade corresponde ao número de valores preenchidos. O mínimo é o menor valor observado, e o máximo é o maior.


**Manual**


In [ ]:
n_preco = len(_validos(preco))
n_area = len(_validos(area))
min_preco, max_preco = min(_validos(preco)), max(_validos(preco))
min_area, max_area = min(_validos(area)), max(_validos(area))
print("Preço — n, mínimo e máximo:", n_preco, min_preco, max_preco)
print("Área — n, mínimo e máximo:", n_area, min_area, max_area)


**Pandas**


In [ ]:
n_preco_pd, n_area_pd = preco.count(), area.count()
min_preco_pd, max_preco_pd = preco.min(), preco.max()
min_area_pd, max_area_pd = area.min(), area.max()
print("Preço — n, mínimo e máximo:", n_preco_pd, min_preco_pd, max_preco_pd)
print("Área — n, mínimo e máximo:", n_area_pd, min_area_pd, max_area_pd)


## 3.3 Média


$$
\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i.
$$

A média é a soma dos valores dividida pela quantidade de observações; $n$ é a quantidade válida e $x_i$ é cada valor.


**Manual**


In [ ]:
def media(valores):
  serie = _validos(valores)
  return sum(serie)/len(serie)

media_preco = media(preco)
media_area = media(area)
print("Preço:", round(media_preco, 2), "US$")
print("Área:", round(media_area, 2), "ft²")


**Pandas**


In [ ]:
def media_pd(valores):
  serie = _validos(valores)
  return serie.mean()

media_preco_pd = media_pd(preco)
media_area_pd = media_pd(area)
print("Preço:", round(media_preco_pd, 2), "US$")
print("Área:", round(media_area_pd, 2), "ft²")


## 3.4 Mediana


$$
\widetilde{x}=
\begin{cases}
x_{\left(\frac{n+1}{2}\right)}, & n\ \text{ímpar},\\[6pt]
\dfrac{x_{\left(\frac{n}{2}\right)}+
x_{\left(\frac{n}{2}+1\right)}}{2}, & n\ \text{par}.
\end{cases}
$$

Após ordenar os valores, a mediana é o valor central ou a média dos dois centrais. $x_{(r)}$ representa o valor na posição $r$ da lista ordenada.


**Manual**


In [ ]:
# Implementa a mediana diretamente a partir dos dados ordenados.
def mediana(valores: pd.Series) -> float:
    """Calcula a mediana depois de ordenar uma cópia dos valores."""
    ordenados = sorted(_validos(valores))
    n = len(ordenados)
    meio = n // 2
    if n % 2 == 1:
        return ordenados[meio]
    return (ordenados[meio - 1] + ordenados[meio]) / 2

mediana_preco = mediana(preco)
mediana_area = mediana(area)
print("Preço:", mediana_preco, "US$")
print("Área:", mediana_area, "ft²")


**Pandas**


In [ ]:
def mediana_pd(valores):
    serie = _validos(valores)
    return serie.median()

mediana_preco_pd = mediana_pd(preco)
mediana_area_pd = mediana_pd(area)
print("Preço:", mediana_preco_pd, "US$")
print("Área:", mediana_area_pd, "ft²")


## 3.5 Moda


$$
\nu(v)=\sum_{i=1}^{n}\mathbb{1}(x_i=v).
$$

$$
\operatorname{Mo}(x)=
\left\{v:\nu(v)=\max_{u}\nu(u)\right\}.
$$

$\nu(v)$ conta quantas vezes o valor $v$ aparece; $\mathbb{1}(x_i=v)$ vale 1 quando os valores são iguais e 0 nos demais casos. A moda reúne todos os valores com a maior frequência.


**Manual**


In [ ]:
def modas(valores):
    serie = _validos(valores)
    freq = serie.value_counts()
    modais = freq[freq == freq.max()].index
    return sorted(modais)

moda_preco = modas(preco)
moda_area = modas(area)
print("Preço:", moda_preco, "US$")
print("Área:", moda_area, "ft²")


**Pandas**


In [ ]:
def modas_pd(valores):
    serie = _validos(valores)
    return serie.mode().to_list()

moda_preco_pd = modas_pd(preco)
moda_area_pd = modas_pd(area)
print("Preço:", moda_preco_pd, "US$")
print("Área:", moda_area_pd, "ft²")


## 3.6 Quartis


$$
I(h)=
\begin{cases}
x_{(1)}, & h\leq1,\\[3pt]
(1-d)x_{(i)}+d x_{(i+1)}, & 1<h<n,\\[3pt]
x_{(n)}, & h\geq n.
\end{cases}
$$

$$
p=\frac{j}{m}.
$$

$$
\begin{gathered}
H_p=1+(n-1)p,\qquad q_p=I(H_p).
\end{gathered}
$$

Para quartis, $m=4$ e $j=1,2,3$, correspondendo a 25%, 50% e 75% dos dados ordenados. Na interpolação, $i=\lfloor h\rfloor$ é a parte inteira da posição e $d=h-i$ é sua parte decimal.


**Manual**


In [ ]:
# Define a interpolação de uma posição teórica
def _interpolar_posicao(ordenados: list[float], h: float) -> float:
    """Interpola uma posição matemática h, indexada a partir de 1."""
    n = len(ordenados)
    if h <= 1:
        return ordenados[0]
    if h >= n:
        return ordenados[-1]

    i = int(h)
    d = h - i
    x_i = ordenados[i - 1]
    x_sup = ordenados[i]
    return x_i + d * (x_sup - x_i)


In [ ]:
# Calcula o quantil pela definição
def quantil(valores: pd.Series, p: float) -> float:
    """Calcula q_p por interpolação linear."""
    if not 0 <= p <= 1:
        raise ValueError("p deve pertencer ao intervalo [0, 1].")

    ordenados = sorted(_validos(valores))
    h = 1 + (len(ordenados) - 1) * p
    return _interpolar_posicao(ordenados, h)

q1_preco = quantil(preco, 0.25)
q2_preco = quantil(preco, 0.50)
q3_preco = quantil(preco, 0.75)
q1_area = quantil(area, 0.25)
q2_area = quantil(area, 0.50)
q3_area = quantil(area, 0.75)
print("Preço — Q1, Q2, Q3:", q1_preco, q2_preco, q3_preco)
print("Área — Q1, Q2, Q3:", q1_area, q2_area, q3_area)


**Pandas**


In [ ]:
def quantil_pd(valores, p):
    if not 0 <= p <= 1:
      raise ValueError('p deve estar entre 0 e 1')
    serie = _validos(valores)
    return serie.quantile(p, interpolation='linear')

q1_preco_pd = quantil_pd(preco, 0.25)
q2_preco_pd = quantil_pd(preco, 0.50)
q3_preco_pd = quantil_pd(preco, 0.75)
q1_area_pd = quantil_pd(area, 0.25)
q2_area_pd = quantil_pd(area, 0.50)
q3_area_pd = quantil_pd(area, 0.75)
print("Preço — Q1, Q2, Q3:", q1_preco_pd, q2_preco_pd, q3_preco_pd)
print("Área — Q1, Q2, Q3:", q1_area_pd, q2_area_pd, q3_area_pd)


## 3.7 Amplitude


$$
R=x_{(n)}-x_{(1)}.
$$

A amplitude é a diferença entre o maior valor, $x_{(n)}$, e o menor, $x_{(1)}$.


**Manual**


In [ ]:
def amplitude(valores):
    serie = _validos(valores)
    return max(serie) - min(serie)

amplitude_preco = amplitude(preco)
amplitude_area = amplitude(area)
print("Preço:", amplitude_preco, "US$")
print("Área:", amplitude_area, "ft²")


**Pandas**


In [ ]:
def amplitude_pd(valores):
    serie = _validos(valores)
    return serie.max() - serie.min()

amplitude_preco_pd = amplitude_pd(preco)
amplitude_area_pd = amplitude_pd(area)
print("Preço:", amplitude_preco_pd, "US$")
print("Área:", amplitude_area_pd, "ft²")


## 3.8 Variância amostral


$$
s^2=\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2.
$$

A variância amostral soma os desvios ao quadrado em relação à média e divide por $n-1$.


**Manual**


In [ ]:
# Implementa a variância amostral com denominador n menos 1.
def variancia(valores: pd.Series) -> float:
    """Calcula a variância amostral com denominador n - 1."""
    observados = _validos(valores, minimo=2)
    media_valor = media(observados)
    soma_quadrados = 0.0
    for valor in observados:
        soma_quadrados += (valor - media_valor) ** 2
    return soma_quadrados / (len(observados) - 1)

variancia_preco = variancia(preco)
variancia_area = variancia(area)
print("Preço:", round(variancia_preco, 2), "US$²")
print("Área:", round(variancia_area, 2), "ft⁴")


**Pandas**


In [ ]:
# Implementa a variância amostral com pandas e ddof igual a 1.
def variancia_pd(valores: pd.Series) -> float:
    """Calcula a variância amostral com pandas e ddof=1."""
    serie = _validos(valores, minimo=2)
    return float(serie.var(ddof=1))

variancia_preco_pd = variancia_pd(preco)
variancia_area_pd = variancia_pd(area)
print("Preço:", round(variancia_preco_pd, 2), "US$²")
print("Área:", round(variancia_area_pd, 2), "ft⁴")


## 3.9 Desvio-padrão amostral


$$
s=\sqrt{s^2}.
$$

O desvio-padrão é a raiz quadrada da variância e tem a mesma unidade dos dados.


**Manual**


In [ ]:
def desvio_padrao(valores):
    return variancia(valores) ** 0.5

desvio_preco = desvio_padrao(preco)
desvio_area = desvio_padrao(area)
print("Preço:", round(desvio_preco, 2), "US$")
print("Área:", round(desvio_area, 2), "ft²")


**Pandas**


In [ ]:
def desvio_padrao_pd(valores):
    return _validos(valores, minimo=2).std(ddof=1)

desvio_preco_pd = desvio_padrao_pd(preco)
desvio_area_pd = desvio_padrao_pd(area)
print("Preço:", round(desvio_preco_pd, 2), "US$")
print("Área:", round(desvio_area_pd, 2), "ft²")


## 3.10 Intervalo interquartil


$$IQR=Q_3-Q_1.$$ A medida representa a extensão dos 50% centrais dos dados.


**Manual**


In [ ]:
iqr_preco = q3_preco - q1_preco
iqr_area = q3_area - q1_area
print("Preço:", iqr_preco, "US$")
print("Área:", iqr_area, "ft²")


**Pandas**


In [ ]:
iqr_preco_pd = preco.quantile(0.75) - preco.quantile(0.25)
iqr_area_pd = area.quantile(0.75) - area.quantile(0.25)
print("Preço:", iqr_preco_pd, "US$")
print("Área:", iqr_area_pd, "ft²")


## 3.11 Resumo das medidas


In [ ]:
medidas = ["Quantidade", "Mínimo", "Máximo", "Média", "Mediana", "Moda", "Q1", "Q2", "Q3",
           "Amplitude", "Variância amostral", "Desvio-padrão amostral", "IQR"]

resumo = pd.DataFrame({
    "Medida": medidas,
    "Preço de venda": [preco.count(), preco.min(), preco.max(), preco.mean(), preco.median(),
                       preco.mode().to_list(), preco.quantile(0.25), preco.quantile(0.50),
                       preco.quantile(0.75), preco.max() - preco.min(), preco.var(ddof=1),
                       preco.std(ddof=1), preco.quantile(0.75) - preco.quantile(0.25)],
    "Área habitável": [area.count(), area.min(), area.max(), area.mean(), area.median(),
                      area.mode().to_list(), area.quantile(0.25), area.quantile(0.50),
                      area.quantile(0.75), area.max() - area.min(), area.var(ddof=1),
                      area.std(ddof=1), area.quantile(0.75) - area.quantile(0.25)]
}).set_index("Medida")
resumo


As medidas de preço estão em US$, e as de área, em ft²; as variâncias estão em US$² e ft⁴. Quantidade representa o número de observações válidas.


## 3.12 Histogramas


In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
preco.plot(kind="hist", bins=30, ax=eixos[0], edgecolor="black", title="Preço de venda (SalePrice)")
area.plot(kind="hist", bins=30, ax=eixos[1], edgecolor="black", title="Área habitável (Gr Liv Area)")
eixos[0].set_xlabel("Preço de venda (US$)")
eixos[1].set_xlabel("Área habitável acima do solo (ft²)")
eixos[0].set_ylabel("Quantidade de imóveis")
eixos[1].set_ylabel("Quantidade de imóveis")
plt.tight_layout()
plt.show()


As duas distribuições têm cauda à direita, com poucos imóveis de preço ou área muito altos. As médias (US$ 180.796,06 e 1.499,69 ft²) superam as medianas (US$ 160.000 e 1.442 ft²).


**Natureza observacional:** Os histogramas resumem vendas observadas, sem intervenção experimental.


**Período e local:** Os registros abrangem Ames, Iowa, de 2006 a 2010; os preços são nominais.


**Representatividade:** Imóveis vendidos nesse contexto não representam automaticamente todos os imóveis ou outras cidades.


**Decisões de preparação:** Foram usadas 30 classes por painel, sem transformação ou exclusão de preços e áreas; outra divisão muda a aparência das barras.


**Valor discrepante e erro:** As caudas mostram valores extremos, que precisam de contexto antes de serem considerados erros.


**Associação e causalidade:** Histogramas mostram cada variável isoladamente e não demonstram que maior área causa maior preço.


## 3.13 Frequências de B


A qualidade da cozinha é ordinal: `Po` (ruim), `Fa` (regular), `TA` (típica/média), `Gd` (boa) e `Ex` (excelente).

$$h_j=\frac{f_j}{n},\qquad p_j=100h_j.$$

A frequência relativa é a quantidade da categoria dividida pelo total de códigos não ausentes; o percentual é essa proporção multiplicada por 100.


In [ ]:
categorias = ["Po", "Fa", "TA", "Gd", "Ex"]
rotulos_originais = ["Ruim (Po)", "Regular (Fa)", "Típica (TA)", "Boa (Gd)", "Excelente (Ex)"]

freq = cozinha.value_counts().reindex(categorias, fill_value=0)
frequencia = freq.to_frame("Quantidade")
frequencia["Relativa"] = cozinha.value_counts(normalize=True).reindex(categorias, fill_value=0)
frequencia["Percentual"] = frequencia["Relativa"] * 100
frequencia.round(4)


## 3.14 Gráfico de barras de B


In [ ]:
frequencia["Quantidade"].plot(kind="bar", figsize=(9, 4), rot=0,
    title="Qualidade da cozinha — Kitchen Quality (Kitchen Qual)")
plt.xticks(range(len(categorias)), rotulos_originais)
plt.xlabel("Qualidade da cozinha")
plt.ylabel("Quantidade de imóveis")
plt.tight_layout()
plt.show()


As categorias típica (`TA`, 1.494 imóveis) e boa (`Gd`, 1.160) concentram a maior parte dos registros. Excelente (`Ex`) tem 205, regular (`Fa`) tem 70 e ruim (`Po`) tem apenas um registro; não há valores ausentes de B.


**Natureza observacional:** A classificação da cozinha foi registrada nas vendas, sem distribuição experimental das qualidades.


**Período e local:** As frequências se referem a Ames, Iowa, no período de 2006 a 2010.


**Representatividade:** A participação das categorias descreve os imóveis vendidos na base; a categoria Po tem apenas uma observação.


**Decisões de preparação:** O gráfico preserva os cinco códigos originais e sua ordem de qualidade; o agrupamento para comparar preços será definido adiante.


**Valor discrepante e erro:** Uma categoria rara não é automaticamente um erro de preenchimento; Po é um código válido na documentação.


**Associação e causalidade:** A frequência de uma qualidade não demonstra sua influência causal sobre o preço.


# 4. Categorias finais de B


Antes de comparar preços, agrupamos `Po` (1 imóvel) e `Fa` (70) em **Baixa (Po/Fa)**: são níveis abaixo da qualidade típica e o grupo resultante tem 71 imóveis. `TA`, `Gd` e `Ex` permanecem separados; essa decisão usa significado e frequência, sem consultar o preço. Os rótulos e a ordem abaixo serão mantidos para a AP3.


In [ ]:
rotulos = {"Po": "Baixa (Po/Fa)", "Fa": "Baixa (Po/Fa)", "TA": "Típica (TA)",
           "Gd": "Boa (Gd)", "Ex": "Excelente (Ex)"}
ordem = ["Baixa (Po/Fa)", "Típica (TA)", "Boa (Gd)", "Excelente (Ex)"]
df["cozinha_grupo"] = df["Kitchen Qual"].map(rotulos)
df["cozinha_grupo"] = pd.Categorical(df["cozinha_grupo"], categories=ordem, ordered=True)


A coluna original `Kitchen Qual` permanece intacta, e `cozinha_grupo` registra a classificação final. A combinação de Po/Fa reduz o problema de comparar um grupo com apenas um imóvel, mas perde a distinção entre essas duas qualidades.


# 5. Análise bivariada


## 5.1 Preço por ano da venda


A tabela apresenta a quantidade de vendas e a mediana do preço em cada ano, em ordem cronológica.


In [ ]:
por_ano = df.groupby("Yr Sold")["SalePrice"]
anual = pd.DataFrame({"Quantidade": por_ano.count(), "Mediana (US$)": por_ano.median()}).sort_index()
anual.index.name = "Ano da venda (Yr Sold)"
anual


In [ ]:
anual["Mediana (US$)"].plot(kind="line", marker="o", figsize=(8, 4),
    title="Mediana do preço de venda (SalePrice) por ano (Yr Sold)")
plt.xticks(anual.index)
plt.xlabel("Ano da venda")
plt.ylabel("Mediana do preço de venda (US$)")
plt.tight_layout()
plt.show()


A mediana anual sobe de US$ 159.500 em 2006 para US$ 165.125 em 2007 e chega a US$ 155.000 em 2010. A linha conecta resumos anuais de diferentes imóveis, com 341 vendas em 2010 e entre 622 e 694 nos demais anos.


**Natureza observacional:** As vendas não foram controladas experimentalmente; a composição dos imóveis pode mudar a cada ano.


**Período e local:** A figura abrange somente Ames, Iowa, de 2006 a 2010, com cinco resumos anuais.


**Representatividade:** Os imóveis vendidos em cada ano e a menor quantidade de 2010 limitam comparações com o mercado geral.


**Decisões de preparação:** A agregação usa medianas anuais e preços nominais, sem ajuste pela inflação; a mediana esconde diferenças internas de cada ano.


**Valor discrepante e erro:** Preços extremos foram mantidos; a mediana é menos sensível a eles, mas essa robustez não comprova a validade dos registros.


**Associação e causalidade:** Composição das vendas, inflação e condições de mercado podem explicar diferenças; a linha não é trajetória do mesmo imóvel, tendência causal nem análise formal de série temporal.


## 5.2 Preço e área construída


A correlação de Pearson mede a associação linear entre área e preço, com valores de −1 a 1. O cálculo considera apenas pares com as duas informações preenchidas.


In [ ]:
pares = df[["Gr Liv Area", "SalePrice"]].dropna()


**Manual**


In [ ]:
x = pares["Gr Liv Area"]
y = pares["SalePrice"]
media_x = media(x)
media_y = media(y)
soma_produtos = 0.0
soma_x = 0.0
soma_y = 0.0
for valor_x, valor_y in zip(x, y):
    desvio_x = valor_x - media_x
    desvio_y = valor_y - media_y
    soma_produtos += desvio_x * desvio_y
    soma_x += desvio_x ** 2
    soma_y += desvio_y ** 2

pearson = soma_produtos / (soma_x * soma_y) ** 0.5
print("Correlação de Pearson:", round(pearson, 4))


**Pandas**


In [ ]:
pearson_pd = pares["Gr Liv Area"].corr(pares["SalePrice"], method="pearson")
print("Correlação de Pearson:", round(pearson_pd, 4))


In [ ]:
pares.plot(kind="scatter", x="Gr Liv Area", y="SalePrice", alpha=0.4, figsize=(8, 5),
           title="Preço de venda (SalePrice) e área habitável (Gr Liv Area)")
plt.xlabel("Área habitável acima do solo (ft²)")
plt.ylabel("Preço de venda (US$)")
plt.tight_layout()
plt.show()


Há associação positiva relativamente forte ($r\approx0,7068$), com forma geral crescente e maior espalhamento dos preços nas áreas maiores. A nuvem principal é contínua, sem grupos nitidamente separados, e alguns imóveis muito grandes têm preços próximos à mediana.


**Natureza observacional:** A área e o preço foram observados nas vendas; outras características dos imóveis não foram controladas.


**Período e local:** A associação corresponde a Ames, Iowa, entre 2006 e 2010, com preços nominais de anos diferentes.


**Representatividade:** O coeficiente descreve os imóveis vendidos na base e não garante a mesma associação em outras populações.


**Decisões de preparação:** Usamos os pares completos e preservamos todos os extremos; neste arquivo não faltam preço nem área.


**Valor discrepante e erro:** Os imóveis de grande área e preço moderado podem alterar o coeficiente de Pearson; sua influência não os torna erros automaticamente.


**Associação e causalidade:** Área e preço estão associados, mas a figura não isola efeitos de localização, conservação ou qualidade e não demonstra causalidade.


## 5.3 Preço por qualidade da cozinha


A tabela apresenta quantidade, mediana, Q1, Q3 e IQR do preço para cada categoria final de qualidade da cozinha.


In [ ]:
por_cozinha = df.groupby("cozinha_grupo", observed=True)["SalePrice"]
resumo_cozinha = pd.DataFrame({"Quantidade": por_cozinha.count(),
                            "Mediana (US$)": por_cozinha.median(),
                            "Q1 (US$)": por_cozinha.quantile(0.25),
                            "Q3 (US$)": por_cozinha.quantile(0.75)}).reindex(ordem)
resumo_cozinha["IQR (US$)"] = resumo_cozinha["Q3 (US$)"] - resumo_cozinha["Q1 (US$)"]
resumo_cozinha.index.name = "Qualidade da cozinha (Kitchen Qual)"
resumo_cozinha


As medianas crescem de US$ 107.500 em Baixa para US$ 136.500 em Típica, US$ 201.000 em Boa e US$ 325.624 em Excelente. Excelente tem o maior IQR (US$ 127.179); os grupos têm tamanhos diferentes e a comparação é apenas descritiva, sem estabelecer causalidade.


# 6. Fontes


- [Enunciado da AP1 — Ricardo Carubbi](https://github.com/carubbi/MQ/blob/main/trabalhos/ap1.md).
- [Ames Housing — Kaggle](https://www.kaggle.com/datasets/shashanknecrothapa/ames-housing-dataset).
- [Documentação das variáveis — Dean De Cock](https://jse.amstat.org/v19n3/decock/DataDocumentation.txt).
- Notebooks de sala `u1_a01`, `u1_a02`, `u1_a05`, `u1_a06`, `u1_a07` e anotações do aluno em `meu_exemplo.ipynb`.
